# OTFLEX Gavin Workflow (Operation-Based Notebook)

This notebook executes the full experimental sequence directly in notebook cells, without graph traversal.

Execution model here:
- Each workflow action is a separate runnable code cell.
- Parameters are declared at the top of each action cell for easy edits.
- Cells are order-agnostic and can be rearranged.
- If something fails, jump to the teardown cell to leave devices in a safe state.

In [1]:
import asyncio
import json
import subprocess
import sys
import time
from pathlib import Path

# Find repo root so imports from src.* work no matter where notebook launches from.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not find repository root containing src/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.adapters.iot_mqtt import (
    PumpMQTT,
    UltraMQTT,
    HeatMQTT,
    ReactorMQTT,
    FurnaceMQTT,
    start_broker_if_needed,
    stop_broker,
    ControllerBeacon,
    _best_effort_all_off,
)
from src.adapters.otflex_adapter import OTFlex
from src.core.opentrons import opentronsClient

print(f"Repo root: {repo_root}")

# Device configs defined directly in this notebook
otflex_cfg = {
    "module": "otflex_runtime.py",
    "controller_ip": "169.254.179.32",
    "deck": {
        "slots": {
            "A1": {"name": "single_electrode_module", "display": "3 electrodes", "labware": "sdl1_single_electrode_tiprack", "file": "data/labware_definitions/sdl1_single_electrode_tiprack.json", "slot_label": "A1"},
            "A2": {"name": "parallel_electrode_module", "display": "4 electrode tool", "labware": "sdl1_parallel_electrode_tiprack", "file": "data/labware_definitions/sdl1_parallel_electrode_tiprack.json", "slot_label": "A2"},
            "A3": {"name": "trash", "display": "trash", "labware": "opentrons_flex_trash", "slot_label": "A3"},
            "B1": {"name": "autreactor", "display": "autreactor", "labware": "actuated_reactor", "file": "data/labware_definitions/actuated_reactor.json", "slot_label": "B1"},
            "B2": {"name": "precursor_solutions", "display": "precursor solutions", "labware": "sdl1_11_vials_20mL", "file": "data/labware_definitions/sdl1_11_vials_20mL.json", "slot_label": "B2"},
            "B3": {"name": "sonicator_bath", "display": "sonicator bath", "labware": "nis_2_sonicator_bath", "file": "data/labware_definitions/nis_2_sonicator_bath.json", "slot_label": "B3"},
            "C3": {"name": "tiprack_1000ul", "display": "1000ul tips", "labware": "opentrons_flex_96_tiprack_1000ul", "slot_label": "C3"},
            "D1": {"name": "substrate_tower", "display": "tower of substrates", "labware": "zou_21_wellplate_4500ul", "file": "data/labware_definitions/zou_21_wellplate_4500ul.json", "slot_label": "D1"}
        },
        "pipettes": {"right": {"model": "p1000_single_flex"}}
    }
}

mqtt_cfg = {
    "broker": "localhost",
    "port": 1883,
    "username": "pyctl-controller",
    "password": "controller",
    "topics": {
        "pumps": "pumps/01",
        "reactor": "reactor/01",
        "furnace": "furnace/01",
        "heat": "heat/01",
        # Keep aligned with MQTT_Demo UltraMQTT base_topic.
        "ultra": "ultra/01",
    },
}

# Critical: flushWell uses OTFlex runtime MQTT helpers, so pass MQTT config into otflex runtime.
otflex_cfg["mqtt"] = mqtt_cfg

opentrons_cfg = {
    "ip": otflex_cfg["controller_ip"],
    "robot": "flex",
}

# Shared runtime objects.
otflex = OTFlex(otflex_cfg, root_dir=repo_root)

beacon = None
pumps = None
ultra = None
heat = None
reactor = None
furnace = None

print("Setup objects created. Run homing cell next, then connect devices.")

Repo root: /mnt/storage/Projects/ACSDL1/AC-OTFlex-monorepo
[OTFlex] Loaded module: /mnt/storage/Projects/ACSDL1/AC-OTFlex-monorepo/src/core/otflex_runtime.py
Setup objects created. Run homing cell next, then connect devices.


## Home Opentrons Robot First

Run this before any workflow action.

This cell inlines the same approach used by `scripts/home_opentrons.py` (directly using `opentronsClient`), without importing that script.

In [17]:
# Homing params (editable)
homing_params = {
    "ip": opentrons_cfg["ip"],
    "robot": opentrons_cfg["robot"],
}

client = opentronsClient(
    strRobotIP=homing_params["ip"],
    strRobot=homing_params["robot"],
)
client.homeRobot()
print(f"Homed {homing_params['robot']} at {homing_params['ip']}")

Homed flex at 169.254.179.32


## Capture Lab Setup Image (Before Connect)

Run this after homing and before connecting devices.

This uses the same SSH capture flow as the verbose Pi camera notebook:
- connect to Raspberry Pi over SSH
- capture an image with Picamera2
- download to `data/out/images`
- rotate the image 180 degrees
- display the image inline
- remove temporary remote image

In [ ]:
# Camera params (editable)
camera_params = {
    "host": "192.168.0.101",
    "username": "sdl1",
    "password": "1144",
    "ssh_port": 22,
    "connect_timeout_s": 8,
    "connect_retries": 3,
    "remote_capture_dir": "/tmp",
    "warmup_s": 2,
    "rotate_degrees": 180,
}

from datetime import datetime
import socket

try:
    import paramiko
except Exception as exc:
    raise ImportError(
        "paramiko is required for SSH camera capture. Install it in the notebook environment."
    ) from exc

try:
    from PIL import Image as PILImage
    from IPython.display import Image as IPyImage, display
except Exception as exc:
    raise ImportError(
        "Pillow and IPython display are required for rotate/display. Install pillow in the notebook environment."
    ) from exc

out_dir = repo_root / "data" / "out" / "images"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"otflex-top_{timestamp}.jpg"
remote_path = f"{camera_params['remote_capture_dir']}/{filename}"
local_path = out_dir / filename

with socket.create_connection(
    (camera_params["host"], int(camera_params["ssh_port"])),
    timeout=3,
    ):
    pass
print(f"SSH port reachable at {camera_params['host']}:{camera_params['ssh_port']}")

ssh = None
try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    for attempt in range(1, int(camera_params["connect_retries"]) + 1):
        try:
            print(f"SSH connect attempt {attempt}/{camera_params['connect_retries']}")
            ssh.connect(
                camera_params["host"],
                port=int(camera_params["ssh_port"]),
                username=camera_params["username"],
                password=camera_params["password"],
                timeout=float(camera_params["connect_timeout_s"]),
                banner_timeout=float(camera_params["connect_timeout_s"]),
                auth_timeout=float(camera_params["connect_timeout_s"]),
            )
            break
        except (socket.timeout, TimeoutError):
            if attempt == int(camera_params["connect_retries"]):
                raise
            print("Timeout; retrying...")

    capture_cmd = f"""python3 << 'EOF'
from picamera2 import Picamera2
import time

picam2 = Picamera2()
config = picam2.create_still_configuration(main={{\"size\": (2028, 1520)}})
picam2.configure(config)
picam2.start()
time.sleep({float(camera_params['warmup_s'])})
picam2.capture_file(\"{remote_path}\")
picam2.close()
print(\"OK\")
EOF
"""

    _, stdout, stderr = ssh.exec_command(capture_cmd, timeout=45)
    exit_code = stdout.channel.recv_exit_status()
    err_text = stderr.read().decode()
    if exit_code != 0:
        raise RuntimeError(f"Pi capture failed: {err_text}")

    sftp = ssh.open_sftp()
    sftp.get(remote_path, str(local_path))
    sftp.close()
    ssh.exec_command(f"rm {remote_path}")

    print(f"Lab setup image saved: {local_path}")

    # Rotate image to match top-down orientation used in verbose notebook.
    with PILImage.open(local_path) as img:
        rotated = img.rotate(int(camera_params["rotate_degrees"]), expand=True)
        rotated.save(local_path)

    print(f"Showing rotated image: {local_path.name}")
    display(IPyImage(filename=str(local_path)))
except Exception:
    raise
finally:
    if ssh is not None:
        ssh.close()

## Connect Devices

This cell mirrors MQTT_Demo MQTT client setup as closely as possible:
- connect OTFlex
- start controller beacon
- connect MQTT devices used by this workflow (pump/ultra/heat/reactor/furnace)
- use `pyctl-*` client IDs for consistent broker/client behavior

In [3]:
# Connection params (editable)
connect_params = {
    "settle_s": 0.8,
    "reset_existing_clients": True,
}

# Keep MQTT settings aligned with MQTT_Demo known-good values.
mqtt_cfg["broker"] = "localhost"
mqtt_cfg["port"] = 1883
mqtt_cfg["username"] = "pyctl-controller"
mqtt_cfg["password"] = "controller"
mqtt_cfg["topics"]["ultra"] = "ultra/01"

if connect_params["reset_existing_clients"]:
    # Re-running this cell with identical client IDs can cause broker-side disconnect thrash.
    for dev in (pumps, ultra, heat, reactor, furnace):
        if dev is not None:
            try:
                dev.disconnect()
            except Exception:
                pass
    if beacon is not None:
        try:
            beacon.stop()
        except Exception:
            pass

await otflex.connect()

beacon = ControllerBeacon(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
    client_id="pyctl-controller",
    status_topic="pyctl/status",
    heartbeat_topic="pyctl/heartbeat",
    heartbeat_interval=5.0,
    keepalive=30,
 )
beacon.start()

common = dict(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
)

pumps = PumpMQTT(**common, base_topic=mqtt_cfg["topics"]["pumps"], client_id="pyctl-pumps")
ultra = UltraMQTT(**common, base_topic=mqtt_cfg["topics"]["ultra"], client_id="pyctl-ultra")
heat = HeatMQTT(**common, base_topic=mqtt_cfg["topics"]["heat"], client_id="pyctl-heat")
reactor = ReactorMQTT(**common, base_topic=mqtt_cfg["topics"]["reactor"], client_id="pyctl-reactor")
furnace = FurnaceMQTT(**common, base_topic=mqtt_cfg["topics"]["furnace"], client_id="pyctl-furnace")

pumps.ensure_connected()
ultra.ensure_connected()
heat.ensure_connected()
reactor.ensure_connected()
furnace.ensure_connected()

# Ensure flushWell and other OTFlex MQTT helpers use known-good notebook clients.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is not None:
    rt.mqtt_pumps = pumps
    rt.mqtt_reactor = reactor
    rt.mqtt_furnace = furnace
    print("Bound OTFlex runtime MQTT handles to notebook MQTT clients.")

time.sleep(float(connect_params["settle_s"]))
print(f"MQTT connected: ultra client={ultra.client_id}, base={ultra.base}")
print("All configured devices connected.")

[OTFlex] Deck layout (normalized):
  slot A1 -> A1 : sdl1_single_electrode_tiprack (single_electrode_module)
  slot A2 -> A2 : sdl1_parallel_electrode_tiprack (parallel_electrode_module)
  slot A3 -> A3 : opentrons_flex_trash (trash)
  slot B1 -> B1 : actuated_reactor (autreactor)
  slot B2 -> B2 : sdl1_11_vials_20mL (precursor_solutions)
  slot B3 -> B3 : nis_2_sonicator_bath (sonicator_bath)
  slot C3 -> C3 : opentrons_flex_96_tiprack_1000ul (tiprack_1000ul)
  slot D1 -> D1 : zou_21_wellplate_4500ul (substrate_tower)
[otflex-pumps] Connecting to localhost:1883 (attempt 1)...
[otflex-pumps] Connected to localhost:1883
[otflex-ultra] Connecting to localhost:1883 (attempt 1)...
[otflex-ultra] Connected to localhost:1883
[otflex-heat] Connecting to localhost:1883 (attempt 1)...
[otflex-reactor] Connecting to localhost:1883 (attempt 1)...
[otflex-heat] Connected to localhost:1883
[otflex-furnace] Connecting to localhost:1883 (attempt 1)...
[otflex-reactor] Connected to localhost:1883
[otf

## Raise Autoreactor (mqttReactor)

Publishes `FORWARD:5000` to the reactor topic and waits for firmware auto-stop timing.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Run xArm Plate-to-Reactor Transfer

This action replaces the fixed pause.

The notebook waits until `src/core/plate2reactor.py` completes, then the next action can lower the reactor.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/plate2reactor.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm transfer action complete.")

## Lower Autoreactor (mqttReactor)

Publishes `REVERSE:8000` and waits for duration alignment.

In [14]:
# Action params (editable)
params = {
    "direction": "reverse",
    "duration_ms": 8000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor lower action complete.")

[pyctl-reactor] Published 'REVERSE:8000' to reactor/01/cmd/1
Reactor lower action complete.


## Transfer Precursor A1 (otflexTransfer)

Transfers precursor from `precursor_solutions.A1` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A1", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## Transfer Precursor A2 (otflexTransfer)

Transfers precursor from `precursor_solutions.A2` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "precursor_solutions", "well": "A2", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## ELECTRODEPOSITION - Operation Context

Initializes shared runtime handles and state used by the electrodeposition operations below.

Run this cell once before running any electrodeposition operation cells.

In [4]:
# Electrochem operation context (shared state + runtime handles)
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if "ec_state" not in globals():
    ec_state = {
        "tool_attached": False,
        "current_well": None,
        "at_ultrasonic": False,
    }

# Default locations used by operation cells; edit as needed.
ec_defaults = {
    "source_labware": "single_electrode_module",
    "source_well": "A1",
    "reactor_labware": "autreactor",
    "ultra_slot": "B3",
    "sonicator_labware": "sonicator_bath",
    "sonicator_well_a": "A1",
    "sonicator_well_b": "A2",
    "sonicator_immersion_mm": 35.0,
    "pipette": "p1000_single_flex",
}

print("Electrochem operation context ready.")
print(f"State: {ec_state}")

Electrochem operation context ready.
State: {'tool_attached': False, 'current_well': None, 'at_ultrasonic': False}


## ELECTRODEPOSITION - Pick Up Electrode Tool

Picks the electrode tool from its storage location and marks the system as tool-attached.

Use this before any well placement, deposition, or ultrasonic cleaning operations.

In [5]:
# Pick electrode tool from source rack
params = {
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "offsetZ": 0.0,
    "pipette": ec_defaults["pipette"],
}

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

await asyncio.to_thread(
    rt.oc.pickUpTip,
    strLabwareName=src_lw,
    strPipetteName=params["pipette"],
    strWellName=params["source_well"],
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=float(params["offsetZ"]),
)

ec_state["tool_attached"] = True
ec_state["current_well"] = None
ec_state["at_ultrasonic"] = False

print(f"Tool picked from {params['source_labware']}.{params['source_well']}")

Tool picked from single_electrode_module.A1


## ELECTRODEPOSITION - Move Tool To Reactor Well

Moves the attached electrode tool to the selected reactor well and positions it for in-well processing.

Use this whenever changing deposition target wells.

In [6]:
# Move attached electrode tool into a reactor well
params = {
    "reactor_labware": ec_defaults["reactor_labware"],
    "target_well": "A1",
    "offsetX": 0.0,
    "offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
descent_mm = float(params["descent_at_well_mm"])
if descent_mm < 0:
    raise ValueError("descent_at_well_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-descent_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["current_well"] = params["target_well"]
ec_state["at_ultrasonic"] = False
print(f"Tool positioned in {params['reactor_labware']}.{params['target_well']} at {descent_mm:.1f} mm depth")

Tool positioned in autreactor.A1 at 25.0 mm depth


## ELECTRODEPOSITION - Run In-Well Deposition

Runs the electrochemistry action while the electrode is already positioned in the reactor well.

This cell is the placeholder location for your final deposition logic.

In [ ]:
# Run electrodeposition while tool is in a well (placeholder)
params = {
    "label": "EDP placeholder",
    "duration_s": 0.0,
}

if ec_state.get("current_well") is None:
    raise RuntimeError("Tool is not in a reactor well. Run the move-to-well cell first.")

print(f"Electrodeposition '{params['label']}' started at well {ec_state['current_well']}")
duration_s = float(params["duration_s"])
if duration_s > 0:
    await asyncio.sleep(duration_s)
print("Electrodeposition placeholder complete.")

## ELECTRODEPOSITION - Move To Ultrasonic Station A

Relocates the attached electrode tool from reactor position to the first ultrasonic cleaning position.

Use this before triggering the first ultrasonic clean cycle.

In [7]:
# Move tool to ultrasonic station A (well A1 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_a"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

Tool moved to sonicator_bath.A1 at 35.0 mm immersion depth


## ELECTRODEPOSITION - Ultrasonic Clean A

Triggers the first ultrasonic cleaning cycle while the electrode tool is held at the ultrasonic station.

Use this between deposition cycles to clean the electrode surface.

In [8]:
# Run ultrasonic transducer (bath A)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

Ultrasonic MQTT client=pyctl-ultra, base=ultra/01, channel=1
[pyctl-ultra] Published 'ON:15000' to ultra/01/cmd/1
[pyctl-ultra] Published 'OFF' to ultra/01/cmd/1
Ultrasonic run complete.


## ELECTRODEPOSITION - Move To Ultrasonic Station B

Relocates the attached electrode tool to the second ultrasonic cleaning position.

Use this when running multi-stage cleaning between deposition operations.

In [9]:
# Move tool to ultrasonic station B (well A2 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_b"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

Tool moved to sonicator_bath.A2 at 35.0 mm immersion depth


## ELECTRODEPOSITION - Ultrasonic Clean B

Triggers the second ultrasonic cleaning cycle at the second cleaning position.

Use this for additional cleaning before returning to deposition or tool storage.

In [10]:
# Run ultrasonic transducer (bath B)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

Ultrasonic MQTT client=pyctl-ultra, base=ultra/01, channel=1
[pyctl-ultra] Published 'ON:15000' to ultra/01/cmd/1
[pyctl-ultra] Published 'OFF' to ultra/01/cmd/1
Ultrasonic run complete.


## ELECTRODEPOSITION - Return Tool

end electrodeposition

In [11]:
# Move to another reactor well, or return tool to source
params = {
    "mode": "drop",  # "drop" or "next_well"
    "next_well": "B1",
    "reactor_labware": ec_defaults["reactor_labware"],
    "reactor_offsetX": 0.0,
    "reactor_offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "source_offsetX": 0.0,
    "source_offsetY": 0.0,
    "return_dZ": 12.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if params["mode"] == "next_well":
    dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
    descent_mm = float(params["descent_at_well_mm"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=0.0,
        intSpeed=int(params["move_speed"]),
    )
    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=-descent_mm,
        intSpeed=int(params["move_speed"]),
    )
    ec_state["current_well"] = params["next_well"]
    ec_state["at_ultrasonic"] = False
    print(f"Tool moved to next well {params['reactor_labware']}.{params['next_well']}")
else:
    src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=10,
        intSpeed=100,
    )
    await asyncio.to_thread(
        rt.oc.dropTip,
        strPipetteName=params["pipette"],
        boolDropInDisposal=False,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strOffsetStart="bottom",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=float(params["return_dZ"]),
    )

    ec_state["tool_attached"] = False
    ec_state["current_well"] = None
    ec_state["at_ultrasonic"] = False
    print(f"Tool dropped back at {params['source_labware']}.{params['source_well']}")

Tool dropped back at single_electrode_module.A1


## Flush Tool Wells (otflexFlushWell)

Flushes wells `A1` and `B1` using pump IDs configured in params.
Each well flush now explicitly starts with OUT pump and ends with OUT pump.

In [16]:
# Action params (editable)
params = {
    "from": {"labware": "single_electrode_module", "well": "C1", "offsetX": 0.0, "offsetY": 0.0, "offsetZ": 0.0},
    "to": {"labware": "autreactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 5.0},
    "pipette": "p1000_single_flex",
    "in_pump_id": 2,
    "out_pump_id": 3,
    "in_time_ms": 2000,
    "out_time_ms": 5000,
    "repeats": 4,
    "purge_ms": 0,
    "return_dZ": 12.0,
}

# Preflight: flushWell relies on OTFlex runtime MQTT pump handle.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or rt.mqtt_pumps is None:
    raise RuntimeError(
        "OTFlex runtime MQTT pumps are not configured. Run the Connect Devices cell first."
    )

await otflex.flushWell(params)
print(
    f"Flush action complete (order per well: OUT -> (IN/OUT)x{params['repeats']} -> OUT; "
    f"in pump {params['in_pump_id']} for {params['in_time_ms']}ms, "
    f"out pump {params['out_pump_id']} for {params['out_time_ms']}ms)."
)

[OTFlex][DEBUG] Flush well operation
[OTFlex][DEBUG] From: single_electrode_module.C1
[OTFlex][DEBUG] To: autreactor.['A1', 'B1']
[OTFlex][DEBUG] Repeats: 4 times
[OTFlex][DEBUG] Pump cycle: in=2 for 2000.0ms, out=3 for 5000.0ms
[OTFlex] Picking up electrode from sdl1_single_electrode_tiprack_A1.C1
[OTFlex] Moving electrode to flush position sdl1_24_wellplate_2664ul_B1.A1 (1/2)
[OTFlex] Starting operation at A1
[OTFlex] Activating MQTT pump channel: 2
[OTFlex] Calling otflex_pump with params: {'pump_id': 2, 'time_ms': 2000.0}
[pyctl-pumps] Published 'ON:2000' to pumps/01/cmd/2
[OTFlex] Waiting 2.000s for pump 2 to finish
[OTFlex] Pump operation completed
[OTFlex] Activating MQTT pump channel: 3
[OTFlex] Calling otflex_pump with params: {'pump_id': 3, 'time_ms': 5000.0}
[pyctl-pumps] Published 'ON:5000' to pumps/01/cmd/3
[OTFlex] Waiting 5.000s for pump 3 to finish
[OTFlex] Pump operation completed
[OTFlex] Activating MQTT pump channel: 2
[OTFlex] Calling otflex_pump with params: {'pump

## Raise Autoreactor Final (mqttReactor)

Final reactor raise before furnace sequence.

In [13]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

[pyctl-reactor] Published 'FORWARD:5000' to reactor/01/cmd/1
Reactor raise action complete.


## Open Furnace Door (mqttFurnace)

Publishes `OPEN:6400` and waits to stay aligned with movement duration.

In [ ]:
# Action params (editable)
# params = {
#     "action": "open",
#     "duration_ms": 6400,
# }

# if params["action"].lower() == "open":
#     await asyncio.to_thread(furnace.open, params["duration_ms"])
# else:
#     await asyncio.to_thread(furnace.close, params["duration_ms"])
# await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
# print("Furnace open action complete.")

## Run xArm Reactor-to-Furnace Transfer

This action replaces the fixed post-open pause.

The notebook waits until `src/core/reactor2furnace.py` completes, then the next action can close the furnace door.

In [ ]:
# Action params (editable)
# params = {
#     "script_relpath": "src/core/reactor2furnace.py",
#     "python_exe": sys.executable,
#     "timeout_s": 0,  # 0 means no timeout
# }

# script_path = (repo_root / params["script_relpath"]).resolve()
# if not script_path.exists():
#     raise FileNotFoundError(f"xArm script not found: {script_path}")

# cmd = [params["python_exe"], str(script_path)]
# print("Running xArm transfer script:", " ".join(cmd))

# run_kwargs = dict(
#     cwd=str(repo_root),
#     text=True,
#     capture_output=True,
#     check=False,
#     timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
# )

# try:
#     result = subprocess.run(cmd, **run_kwargs)
# except subprocess.TimeoutExpired as exc:
#     raise TimeoutError(
#         f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
#     ) from exc

# if result.stdout:
#     print(result.stdout)
# if result.stderr:
#     print(result.stderr)

# if result.returncode != 0:
#     raise RuntimeError(
#         f"xArm transfer failed with exit code {result.returncode}: {script_path}"
#     )

# print("xArm reactor-to-furnace action complete.")

## Close Furnace Door (mqttFurnace)

Publishes `CLOSE:10000` and waits to align with door motion.

In [ ]:
# Action params (editable)
# params = {
#     "action": "close",
#     "duration_ms": 10000,
# }

# if params["action"].lower() == "open":
#     await asyncio.to_thread(furnace.open, params["duration_ms"])
# else:
#     await asyncio.to_thread(furnace.close, params["duration_ms"])
# await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
# print("Furnace close action complete.")

## End Marker

No hardware action. This marks the workflow end.

In [ ]:
print("Workflow sequence reached end marker.")

## Teardown - Safe Shutdown

Run this at the end, or immediately if any step errors.

This mirrors the runner's best-effort shutdown intent:
- turn off active MQTT outputs
- disconnect MQTT clients and beacon
- disconnect OTFlex

In [ ]:
try:
    _best_effort_all_off(pumps=pumps, ultra=ultra, heat=heat, reactor=reactor, furnace=furnace)
except Exception as exc:
    print(f"Best-effort OFF warning: {exc}")

for dev, name in ((pumps, "pumps"), (ultra, "ultra"), (heat, "heat"), (reactor, "reactor"), (furnace, "furnace")):
    if dev is not None:
        try:
            dev.disconnect()
        except Exception as exc:
            print(f"Disconnect warning ({name}): {exc}")

if beacon is not None:
    try:
        beacon.stop()
    except Exception as exc:
        print(f"Beacon stop warning: {exc}")

await otflex.disconnect()
print("Teardown complete.")